<a href="https://colab.research.google.com/github/petrovortex/foundations_of_ml_course/blob/main/hometask_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
fall_ml_6_mipt_2025_path = kagglehub.competition_download('fall-ml-6-mipt-2025')

print('Data source import complete.')

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/kaggle/input/fall-ml-6-mipt-2025/train.csv')
X_test = pd.read_csv('/kaggle/input/fall-ml-6-mipt-2025/test.csv')

In [ ]:
y = df['TARGET']
X = df.drop(labels=['TARGET'], axis=1)

## 1. EDA

In [ ]:
explorator = DataExplorator()

In [ ]:
explorator.get_info(df)

,Type,Count,Nunique,Nulls,Most frequent
Колво_отсроченных_платежей,object,70000,529,4925,"[10, 19, 17]"
Минимальный_платеж,object,70000,3,0,"[Yes, No, NM]"
Месячный_баланс,object,70000,69161,833,"[__-333333333333333333333333333__, 415.9340612..."
Задержка_платежа_дни,int64,70000,73,0,"[15, 13, 8]"
ID_записи,object,70000,70000,0,"[0x1e979, 0xa03f, 0x834d]"
ID_клиента,object,70000,12496,3456,"[CUS_0xa12, CUS_0x7e61, CUS_0x872d]"
Годовой_доход,object,70000,17304,0,"[36585.12, 20867.67, 32543.38]"
Сумма_ежемесячных_выплат,float64,70000,13940,0,"[0.0, 43.28294046828422, 72.87631763689136]"
Процентная_ставка,int64,70000,1269,0,"[8, 5, 6]"
Колво_займов,object,70000,315,0,"[3, 2, 4]"


In [ ]:
df['Тип_кредита'].value_counts()

## 2. Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

feature_map = {
    'Колво_отсроченных_платежей': 'Num_Delayed_Payment',
    'Минимальный_платеж': 'Minimum_Payment_Status',
    'Месячный_баланс': 'Monthly_Balance',
    'Задержка_платежа_дни': 'Delay_from_due_date',
    'Годовой_доход': 'Annual_Income',
    'Сумма_ежемесячных_выплат': 'Total_EMI_per_month',
    'Процентная_ставка': 'Interest_Rate',
    'Колво_займов': 'Num_of_Loan',
    'Колво_банковских_счетов': 'Num_Bank_Accounts',
    'Оставшийся_долг': 'Outstanding_Debt',
    'Коэффициент_использования_кредита': 'Credit_Utilization_Ratio',
    'Тип_кредита': 'Type_of_Loan',
    'Колво_кредитных_карт': 'Num_Credit_Card',
    'Месяц': 'Month',
    'Колво_кредитных_запросов': 'Num_Credit_Inquiries',
    'Профессия': 'Occupation',
    'Кредитный_микс': 'Credit_Mix',
    'Месячная_зарплата': 'Monthly_Inhand_Salary',
    'Возраст_кредитной_истории': 'Credit_History_Age',
    'Клиент_Инфо': 'Age',
    'Сумма_инвестиций': 'Investment_Amount',
    'Платежное_поведение': 'Payment_Behaviour',
    'Изменение_кредитного_лимита': 'Changed_Credit_Limit',
    'ID_клиента': 'client_ID'
}

class Preprocessor:
    def __init__(self, feature_map):

        self.saved_medians = {}
        self.feature_map = feature_map

    def fit(self, X, y=None):
        df_temp = X.copy()
        df_temp = df_temp.rename(columns=self.feature_map)
        df_temp = self._clean_raw_data(df_temp)
        df_temp = self._engineer_features(df_temp)

        for col in self.cols_median_impute:
            self.saved_medians[col] = df_temp[col].median()

        return self

    def transform(self, X, y=None, is_tabicl=True):
        df = X.copy()

        df = df.rename(columns=self.feature_map)

        df = self._clean_raw_data(df)
        df = self._engineer_features(df)
        df = self._apply_imputation(df)

        if is_tabicl:
            df = self._finalize_for_tabicl(df)

        return df

    def _clean_raw_data(self, df):
        cols_to_drop = ['ID_записи', 'SSN']
        df = df.drop(columns=cols_to_drop, errors='ignore')

        def clean_delayed_payment(x):
            try:
                val = float(str(x).replace('_', ''))
                return val if 0 <= val <= 28 else np.nan
            except:
                return np.nan

        def clean_float_underscore(x):
            try: return float(str(x).replace('_', ''))
            except: return np.nan

        def clean_num_loans(x):
            try:
                val = float(str(x).replace('_', ''))
                return val if -100 < val <= 9 else np.nan
            except: return np.nan

        def parse_history_age(x):
            if pd.isna(x): return np.nan
            x = str(x)
            years = re.search(r'(\d+)\s*Year', x)
            months = re.search(r'(\d+)\s*Month', x)
            if not years and not months: return np.nan
            y_val = int(years.group(1)) if years else 0
            m_val = int(months.group(1)) if months else 0
            return y_val * 12 + m_val

        def parse_client_info(x):
            if pd.isna(x): return np.nan
            parts = str(x).split('|')
            if len(parts) > 1:
                try: return float(parts[1])
                except: return np.nan
            return np.nan

        df['Num_Delayed_Payment'] = df['Num_Delayed_Payment'].apply(clean_delayed_payment)

        df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'], errors='coerce')

        df['Annual_Income'] = df['Annual_Income'].apply(clean_float_underscore)
        df['Outstanding_Debt'] = df['Outstanding_Debt'].apply(clean_float_underscore)

        df['Interest_Rate'] = pd.to_numeric(df['Interest_Rate'], errors='coerce')
        df.loc[df['Interest_Rate'] > 34, 'Interest_Rate'] = np.nan

        df['Num_of_Loan'] = df['Num_of_Loan'].apply(clean_num_loans)

        df['Num_Bank_Accounts'] = pd.to_numeric(df['Num_Bank_Accounts'], errors='coerce')
        df.loc[df['Num_Bank_Accounts'] > 10, 'Num_Bank_Accounts'] = np.nan

        df['Num_Credit_Card'] = pd.to_numeric(df['Num_Credit_Card'], errors='coerce')
        df.loc[df['Num_Credit_Card'] > 10, 'Num_Credit_Card'] = np.nan

        df['Type_of_Loan'] = df['Type_of_Loan'].astype(str).str.replace(", and", ",", regex=False)

        df['Num_Credit_Inquiries'] = pd.to_numeric(df['Num_Credit_Inquiries'], errors='coerce')
        df.loc[df['Num_Credit_Inquiries'] > 17, 'Num_Credit_Inquiries'] = np.nan

        df['Occupation'] = df['Occupation'].replace('_______', np.nan)

        df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)

        df['Credit_History_Age'] = df['Credit_History_Age'].apply(parse_history_age)

        df['Age'] = df['Age'].apply(parse_client_info)

        df['Investment_Amount'] = pd.to_numeric(df['Investment_Amount'], errors='coerce')

        df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)

        df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'], errors='coerce')

        return df

    def _engineer_features(self, df):
        df['DTI_Ratio'] = df['Outstanding_Debt'] / df['Annual_Income']

        df['EMI_Income_Ratio'] = df['Total_EMI_per_month'] / df['Monthly_Inhand_Salary']

        inv = df['Investment_Amount'].replace(0, 0.01)
        df['Debt_Investment_Ratio'] = df['Outstanding_Debt'] / inv

        loans = df['Num_of_Loan'].replace(0, 1)
        df['Avg_Debt_per_Loan'] = df['Outstanding_Debt'] / loans

        df['Annual_Monthly_Income_Diff'] = df['Annual_Income'] - (12 * df['Monthly_Inhand_Salary'])

        df['Disposable_Income'] = df['Monthly_Inhand_Salary'] - df['Total_EMI_per_month']

        df['Delay_Intensity'] = df['Delay_from_due_date'] * df['Num_Delayed_Payment']

        df['Loan_Type_Count'] = df['Type_of_Loan'].apply(lambda x: len(str(x).split(',')) if x != 'nan' and x != 'nan' else 0)

        df['Has_Payday_Loan'] = df['Type_of_Loan'].apply(lambda x: 'Yes' if 'Payday' in str(x) else 'No')
        df['Has_Mortgage'] = df['Type_of_Loan'].apply(lambda x: 'Yes' if 'Mortgage' in str(x) else 'No')

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        return df

    def _apply_imputation(self, df):
        for col in df.columns.to_list():
            df[col] = df[col].fillna('Unknown')
        return df

    def _finalize_for_tabicl(self, df):
        float_cols = df.select_dtypes(include=['float']).columns
        def tabicl_format(x, sig_fig=3):
            if pd.isnull(x):
                return x

            abs_x = abs(x)

            if abs_x == 0:
                return "0"
            elif abs_x >= 1000000:
                return f"{x/1000000:.{sig_fig-1}} million"
            elif abs_x >= 1000:
                return f"{x:,.0f}"
            elif abs_x < 0.001:
                return f"{x:.{sig_fig+3}g}"
            else:
                return f"{x:.{sig_fig}g}"

        for col in float_cols:
            df[col] = df[col].apply(lambda x: tabicl_format(x, 3))

        df = df.fillna('nan')
        df = df.astype(str)
        df = df.replace(['<NA>', 'nan', 'NaN', 'None', 'nan'], 'nan')
        return df

In [ ]:
preprocessor = Preprocessor(feature_map=feature_map)

X_train_tabicl = preprocessor.transform(X_train, is_tabicl=True)
X_test_tabicl = preprocessor.transform(X_test, is_tabicl=True)

X_train_cb = preprocessor.transform(X_train, is_tabicl=False)
X_test_cb = preprocessor.transform(X_test, is_tabicl=False)

## 3. Training and testing